Reference:

    - Perceiver IO: https://arxiv.org/abs/2107.14795

In [ ]:
# import os
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import pytorch_lightning as pl
# from torch.utils.data import DataLoader
# from typing import List, Tuple, Dict
# from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR

# # --- Make sure these imports are correct for your project structure ---
# from config import project_paths, simd_r_drive_server_config
# from models.pytorch.narrative_stack.stage2.dataset import (
#     Stage2StackDataset,
#     stage2_collate_stacks,
# )

# # ======== DIAGNOSTIC CONFIGURATION ========
# CONFIG = {
#     "lr": 5e-5, 
#     "batch_size": 8,
#     "grad_clip_val": 1.0,
#     "loss_alpha": 0.5,
#     "dropout_rate": 0.1,
#     "weight_decay": 1e-5,
#     "warmup_steps": 200,
#     "input_dim": 256,
#     "latent_dim": 256,
#     "num_latents": 64,
#     "encoder_depth": 2,
#     "query_dim": 256,
#     "decoder_depth": 2,
# }

# # ======== BUILDING BLOCKS FOR DIAGNOSTIC TEST (FINAL FIX) ========

# class PerceiverStackEncoder(nn.Module):
#     """ The encoder for a single stack. """
#     def __init__(self, input_dim: int, latent_dim: int, num_latents: int, depth: int, dropout_rate: float):
#         super().__init__()
#         self.latents = nn.Parameter(torch.randn(num_latents, latent_dim))
#         self.cross_attn = nn.MultiheadAttention(embed_dim=latent_dim, num_heads=4, batch_first=True, dropout=dropout_rate)
#         self.cross_proj = nn.Linear(input_dim, latent_dim)
#         self.query_norm = nn.LayerNorm(latent_dim)
#         self.kv_norm = nn.LayerNorm(latent_dim)
#         self.blocks = nn.ModuleList(
#             [
#                 nn.TransformerEncoderLayer(
#                     d_model=latent_dim, nhead=4, dim_feedforward=latent_dim * 4,
#                     batch_first=True, activation=F.gelu, norm_first=True,
#                     dropout=dropout_rate
#                 ) for _ in range(depth)
#             ]
#         )
#         self.output_norm = nn.LayerNorm(latent_dim)

#     def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
#         B, N, latent_dim = x.shape[0], x.shape[1], self.latents.size(-1)
#         latents = self.latents.unsqueeze(0).repeat(B, 1, 1)
#         x_proj = self.cross_proj(x)
#         latents_norm = self.query_norm(latents)
#         x_proj_norm = self.kv_norm(x_proj)

#         # The key_padding_mask is True for padded values that should be ignored.
#         key_padding_mask = ~mask

#         # --- FINAL FIX: Prevent softmax(all -inf) which results in NaN ---
#         # Identify items in the batch that are fully padded (i.e., their mask has no True values).
#         fully_padded_items = ~mask.any(dim=1)
        
#         # If any such items exist...
#         if fully_padded_items.any():
#             # ...for those specific items, set the first element of their key_padding_mask to False.
#             # This "tricks" the softmax into thinking there's one valid item to attend to,
#             # preventing the computation of softmax([-inf, -inf, ...]).
#             key_padding_mask[fully_padded_items, 0] = False

#         attn_output, _ = self.cross_attn(
#             query=latents_norm, key=x_proj_norm, value=x_proj_norm, key_padding_mask=key_padding_mask
#         )
        
#         latents = latents + attn_output
#         for block in self.blocks:
#             latents = block(latents)
#         return self.output_norm(latents).mean(dim=1)

# class PerceiverDecoder(nn.Module):
#     """ A simplified decoder for a single context vector. """
#     def __init__(self, context_dim: int, query_dim: int, output_dim: int, depth: int, dropout_rate: float):
#         super().__init__()
#         self.context_proj = nn.Linear(context_dim, query_dim)
#         self.cross_attn = nn.MultiheadAttention(embed_dim=query_dim, num_heads=4, batch_first=True, dropout=dropout_rate)
#         self.query_norm = nn.LayerNorm(query_dim)
#         self.context_norm = nn.LayerNorm(query_dim)
#         self.blocks = nn.ModuleList(
#             [
#                 nn.TransformerEncoderLayer(
#                     d_model=query_dim, nhead=4, dim_feedforward=query_dim * 4,
#                     batch_first=True, activation=F.gelu, norm_first=True,
#                     dropout=dropout_rate
#                 ) for _ in range(depth)
#             ]
#         )
#         self.output_head = nn.Linear(query_dim, output_dim)

#     def forward(self, context_vector: torch.Tensor, output_query: torch.Tensor) -> torch.Tensor:
#         projected_context = self.context_proj(context_vector)
#         latent = projected_context.unsqueeze(1).expand(-1, output_query.size(1), -1)
#         query_norm = self.query_norm(output_query)
#         latent_norm = self.context_norm(latent)
#         attn_output, _ = self.cross_attn(query=query_norm, key=latent_norm, value=latent_norm)
#         x = output_query + attn_output
#         for block in self.blocks:
#             x = block(x)
#         return self.output_head(x)


# # ======== SINGLE STACK TESTER MODULE ========
# class SingleStackTester(pl.LightningModule):
#     """A minimal autoencoder for a single stack to test if the core components can learn."""
#     def __init__(self, hparams: dict):
#         super().__init__()
#         self.save_hyperparameters(hparams)
        
#         self.encoder = PerceiverStackEncoder(
#             input_dim=self.hparams.input_dim,
#             latent_dim=self.hparams.latent_dim,
#             num_latents=self.hparams.num_latents,
#             depth=self.hparams.encoder_depth,
#             dropout_rate=self.hparams.dropout_rate
#         )
        
#         self.decoder = PerceiverDecoder(
#             context_dim=self.hparams.latent_dim,
#             query_dim=self.hparams.query_dim,
#             output_dim=self.hparams.input_dim,
#             depth=self.hparams.decoder_depth,
#             dropout_rate=self.hparams.dropout_rate
#         )

#         self.balance_embedding = nn.Embedding(num_embeddings=3, embedding_dim=self.hparams.query_dim)
#         self.period_embedding = nn.Embedding(num_embeddings=3, embedding_dim=self.hparams.query_dim)
#         self.loss_fn = nn.MSELoss()

#     def forward(self, stack, mask, balance, period):
#         latent_vector = self.encoder(stack, mask)
        
#         balance_embed = self.balance_embedding(balance)
#         period_embed = self.period_embedding(period)
#         query = balance_embed + period_embed
        
#         reconstruction = self.decoder(latent_vector, query)
#         return reconstruction

#     def training_step(self, batch, batch_idx):
#         stacks, masks, balance_batch, period_batch = batch
        
#         # We only use the first category for this test
#         stack_0, mask_0, balance_0, period_0 = stacks[0], masks[0], balance_batch[0], period_batch[0]
        
#         recon = self.forward(stack_0, mask_0, balance_0, period_0)
#         original = stack_0[mask_0]
#         recon = recon[mask_0]

#         if original.numel() == 0:
#             return None

#         mse_loss = self.loss_fn(recon, original)
#         cosine_loss = (1 - F.cosine_similarity(recon, original, dim=1, eps=1e-8)).mean()
#         loss = self.hparams.loss_alpha * mse_loss + (1 - self.hparams.loss_alpha) * cosine_loss
        
#         if not torch.isfinite(loss):
#             print(f"WARNING: Loss is NaN or Inf at batch {batch_idx}. Skipping update.")
#             return None

#         self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
#         return loss

#     def configure_optimizers(self):
#         optimizer = torch.optim.AdamW(
#             self.parameters(), 
#             lr=self.hparams.lr,
#             weight_decay=self.hparams.weight_decay
#         )
        
#         try:
#             total_steps = self.trainer.estimated_stepping_batches
#         except AttributeError:
#             total_steps = 100_000 

#         warmup_scheduler = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0, total_iters=self.hparams.warmup_steps)
#         main_scheduler_t_max = max(1, total_steps - self.hparams.warmup_steps)
#         main_scheduler = CosineAnnealingLR(optimizer, T_max=main_scheduler_t_max, eta_min=1e-7)

#         lr_scheduler = torch.optim.lr_scheduler.SequentialLR(
#             optimizer, schedulers=[warmup_scheduler, main_scheduler], milestones=[self.hparams.warmup_steps]
#         )
        
#         return {
#             "optimizer": optimizer,
#             "lr_scheduler": {
#                 "scheduler": lr_scheduler,
#                 "interval": "step",
#             },
#         }

# # ======== MAIN EXECUTION ========
# def run_diagnostic():
#     print("==============================================")
#     print("     RUNNING SINGLE STACK DIAGNOSTIC TEST     ")
#     print("==============================================")
#     print("Goal: Confirm the building blocks are stable and can learn.")
#     print("----------------------------------------------")

#     # --- 1. Set up DataLoader ---
#     train_loader = DataLoader(
#         Stage2StackDataset(
#             simd_r_drive_server_config=simd_r_drive_server_config,
#             shuffle=True,
#             lookup_batch_size=64,
#         ),
#         batch_size=CONFIG["batch_size"],
#         collate_fn=stage2_collate_stacks,
#         num_workers=0, # Use 0 workers for simplicity in debugging
#     )

#     # --- 2. Create Model ---
#     model = SingleStackTester(CONFIG)

#     # --- 3. Set up Trainer ---
#     trainer = pl.Trainer(
#         max_epochs=5, # Run for a few epochs to see if loss moves
#         accelerator="auto",
#         devices=1,
#         enable_progress_bar=True,
#         enable_checkpointing=False,
#         logger=True, 
#         gradient_clip_val=CONFIG["grad_clip_val"],
#     )

#     # --- 4. Run the Test ---
#     try:
#         print("Starting trainer.fit()...")
#         trainer.fit(model, train_dataloaders=train_loader)
#         print("\n--- Diagnostic Finished ---")
#     except Exception as e:
#         print(f"\nFATAL: Training crashed during diagnostic. Error: {e}")

# if __name__ == "__main__":
#     run_diagnostic()


# Data Sanity Checks

In [ ]:
"""
Quick sanity check that Stage2StackDataset yields data.

This cell:
  1) Builds train/val datasets with shuffle=False.
  2) Uses a single-process DataLoader (workers=0).
  3) Fetches one batch from each and logs a short summary.

If you still see a NoneType error from __iter__, your dataset is trying
to use range(self._count) before _count is initialized.
"""

import logging
import traceback
from torch.utils.data import DataLoader
import torch

from config import simd_r_drive_server_config
from models.pytorch.narrative_stack.stage2.dataset import (
    Stage2StackDataset,
    stage2_collate_stacks,
)

# ------------------------------------------------------------------ #
# Logging
# ------------------------------------------------------------------ #
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
log = logging.getLogger("stage2_sanity")


def _summarize_obj(name, obj):
    """
    Log a compact summary of a Python object or tensor structure.
    """
    try:
        import numpy as _np  # optional, only for isinstance checks
    except Exception:
        _np = None

    if isinstance(obj, torch.Tensor):
        log.info(
            "%s: Tensor shape=%s dtype=%s device=%s",
            name, tuple(obj.shape), str(obj.dtype), str(obj.device),
        )
        return

    if _np is not None and isinstance(obj, _np.ndarray):
        log.info(
            "%s: ndarray shape=%s dtype=%s",
            name, obj.shape, str(obj.dtype),
        )
        return

    if isinstance(obj, dict):
        keys = list(obj.keys())
        log.info("%s: dict keys=%s", name, keys[:10])
        for k in keys[:3]:
            _summarize_obj("%s[%r]" % (name, k), obj[k])
        return

    if isinstance(obj, (list, tuple)):
        log.info("%s: %s len=%d", name, type(obj).__name__, len(obj))
        if len(obj) > 0:
            _summarize_obj("%s[0]" % name, obj[0])
        return

    log.info("%s: %s repr=%r", name, type(obj).__name__, obj)


def _get_one_batch(tag, ds, bs=2):
    """
    Build a DataLoader and fetch one batch, logging its structure.
    """
    loader = DataLoader(
        ds,
        batch_size=bs,
        shuffle=False,
        num_workers=0,
        collate_fn=stage2_collate_stacks,
        persistent_workers=False,
    )

    it = iter(loader)
    try:
        batch = next(it)
    except StopIteration:
        log.info("%s: dataset produced no items", tag)
        return False
    except Exception as e:
        log.error("%s: exception while iterating: %s", tag, str(e))
        tb = traceback.format_exc(limit=3)
        log.error("%s: traceback:\n%s", tag, tb)
        return False

    _summarize_obj("%s.batch" % tag, batch)
    return True


# ------------------------------------------------------------------ #
# Create datasets with shuffle=False to avoid index-based shuffling.
# ------------------------------------------------------------------ #
train_ds = Stage2StackDataset(
    simd_r_drive_server_config=simd_r_drive_server_config,
    shuffle=False,
    lookup_batch_size=32,
)

val_ds = Stage2StackDataset(
    simd_r_drive_server_config=simd_r_drive_server_config,
    shuffle=False,
    lookup_batch_size=32,
)

ok_train = _get_one_batch("train", train_ds, bs=2)
ok_val = _get_one_batch("val", val_ds, bs=2)

if ok_train and ok_val:
    log.info("Success: both train and val yielded at least one batch.")
elif ok_train:
    log.info("Partial: train yielded data, val did not.")
elif ok_val:
    log.info("Partial: val yielded data, train did not.")
else:
    log.info("Failure: neither train nor val yielded a batch.")


In [ ]:
"""
Probe Stage 2 data presence without touching dataset/base classes.

It prints:
  - row_count from UsGaapStore.get_stage2_row_count()
  - for the first K rows: how many categories are non-empty
  - shapes for the first row that contains any data
"""

from config import simd_r_drive_server_config
from simd_r_drive_ws_client import DataStoreWsClient
from us_gaap_store import UsGaapStore, STAGE2_CATEGORY_STACKS

def _safe_int(x):
    """Return int(x) or 0 on error/None."""
    try:
        return int(x)
    except Exception:
        return 0

client = DataStoreWsClient(
    simd_r_drive_server_config.host,
    simd_r_drive_server_config.port,
)
store = UsGaapStore(client)

row_count = store.get_stage2_row_count()
rc = _safe_int(row_count)
print("Stage2 row_count:", rc)

if rc <= 0:
    print("No Stage 2 rows reported. Nothing to sample.")
else:
    K = min(64, rc)
    rows = list(range(K))
    print("Sampling first", K, "rows...")

    cell_map = store.get_stage2_category_stacks_cell_indices_by_rows(rows)
    latents = store.get_cached_stage2_category_stacks_latents_by_cell_indices(
        cell_map
    )

    print("Fetched", len(latents), "rows of latent maps.")

    any_rows_with_data = 0
    per_cat_counts = {c: 0 for c in STAGE2_CATEGORY_STACKS}
    example_shapes = None

    for row in latents:
        row_has_data = False
        row_shapes = {}
        for cat in STAGE2_CATEGORY_STACKS:
            arr = row.get(cat)
            n = len(arr) if arr is not None else 0
            if n > 0 and hasattr(arr, "shape"):
                row_has_data = True
                per_cat_counts[cat] += 1
                row_shapes[cat] = tuple(arr.shape)
            else:
                row_shapes[cat] = (0, 0)
        if row_has_data:
            any_rows_with_data += 1
            if example_shapes is None:
                example_shapes = row_shapes

    print("Rows with any non-empty category:", any_rows_with_data, "/", len(latents))
    for cat in STAGE2_CATEGORY_STACKS:
        print("Non-empty rows for %-18s: %d" % (cat, per_cat_counts[cat]))

    if example_shapes is not None:
        print("Example row shapes by category:")
        for cat in STAGE2_CATEGORY_STACKS:
            print("  %-18s -> %r" % (cat, example_shapes[cat]))
    else:
        print("No non-empty categories found in the sampled rows.")

# Cleanup
del store
del client


# Hyperparameter Tuning

In [ ]:
import os
import torch
import optuna
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import EarlyStopping
from torch.utils.data import DataLoader
from pytorch_lightning.callbacks import ModelCheckpoint

# --- Make sure these imports are correct for your project structure ---
from config import project_paths, simd_r_drive_server_config
from models.pytorch.narrative_stack.stage2 import Stage2Autoencoder
from models.pytorch.narrative_stack.stage2.dataset import (
    Stage2StackDataset,
    stage2_collate_stacks,
)

# ======== TUNING CONFIGURATION ========
OUTPUT_PATH = project_paths.python_data / "stage2_tuning"
os.makedirs(OUTPUT_PATH, exist_ok=True)
OPTUNA_DB_PATH = f"sqlite:///{OUTPUT_PATH / 'stage2_study.db'}"

# Training settings for each trial
EPOCHS = 5
PATIENCE = 5
N_TRIALS = 20
LOOKUP_BATCH_SIZE = 32

NUM_WORKERS = 2

# ======== OPTUNA OBJECTIVE FUNCTION ========


def objective(trial: optuna.trial.Trial) -> float:
    """
    Single training run ("trial") invoked by Optuna with different
    hyperparameters.
    """
    torch.autograd.set_detect_anomaly(True)

    # --- 1. Suggest Hyperparameters ---
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [4])
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3,
                                       log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.05, 0.3,
                                       step=0.05)
    loss_alpha = trial.suggest_float("loss_alpha", 0.2, 0.8)
    warmup_steps = trial.suggest_categorical("warmup_steps",
                                             [250, 500, 1000, 2000])

    # Encoder architecture
    encoder_latent_dim = trial.suggest_categorical("encoder_latent_dim",
                                                   [128, 192])
    num_latents = trial.suggest_categorical("num_latents", [64, 128])
    encoder_depth = trial.suggest_int("encoder_depth", 3, 6)

    # Decoder and shared architecture
    shared_latent_dim = trial.suggest_categorical("shared_latent_dim",
                                                  [128, 256, 512, 768,
                                                   1024])
    query_dim = trial.suggest_categorical("query_dim", [192, 256])
    decoder_depth = trial.suggest_int("decoder_depth", 3, 6)

    checkpoint_callback = ModelCheckpoint(
        monitor="val_loss_epoch",
        mode="min",
        save_top_k=1,
        filename="trial-{epoch:02d}-{val_loss_epoch:.4f}",
    )

    # TODO: pin_memory=True? Probably not for MPS.

    # --- 2. Set up DataLoaders ---
    train_loader = DataLoader(
        Stage2StackDataset(
            simd_r_drive_server_config=simd_r_drive_server_config,
            shuffle=True,
            lookup_batch_size=LOOKUP_BATCH_SIZE,
        ),
        batch_size=batch_size,
        collate_fn=stage2_collate_stacks,
        num_workers=NUM_WORKERS,
        persistent_workers=True if NUM_WORKERS > 0 else False,
    )

    val_loader = DataLoader(
        Stage2StackDataset(
            simd_r_drive_server_config=simd_r_drive_server_config,
            shuffle=False,
            lookup_batch_size=LOOKUP_BATCH_SIZE,
        ),
        batch_size=batch_size,
        collate_fn=stage2_collate_stacks,
        num_workers=NUM_WORKERS,
        persistent_workers=True if NUM_WORKERS > 0 else False,
    )

    # --- 3. Create Model with Suggested Hyperparameters ---
    model = Stage2Autoencoder(
        loss_alpha=loss_alpha,
        dropout_rate=dropout_rate,
        weight_decay=weight_decay,
        warmup_steps=warmup_steps,
        encoder_latent_dim=encoder_latent_dim,
        num_latents=num_latents,
        encoder_depth=encoder_depth,
        shared_latent_dim=shared_latent_dim,
        query_dim=query_dim,
        decoder_depth=decoder_depth,
        batch_size=batch_size,
        lr=lr,
    )

    # --- 4. Set up Trainer ---
    logger = TensorBoardLogger(
        OUTPUT_PATH,
        name="tuning_logs",
        version=f"trial_{trial.number}",
    )
    early_stop_callback = EarlyStopping(
        monitor="val_loss_epoch",
        patience=PATIENCE,
        verbose=True,
        mode="min",
    )

    trainer = pl.Trainer(
        accelerator="cpu",
        devices=1,
        max_epochs=EPOCHS,
        logger=logger,
        callbacks=[early_stop_callback, checkpoint_callback],
        enable_progress_bar=True,
        enable_checkpointing=True,
        gradient_clip_val=1.0,
        # IterableDataset requires 1.0 or int, not a fraction.
        limit_train_batches=1.0,
        limit_val_batches=1.0,
        # Avoid early validation pass triggering dataset init twice.
        num_sanity_val_steps=0,
    )

    # --- 5. Train the Model ---
    try:
        trainer.fit(model,
                    train_dataloaders=train_loader,
                    val_dataloaders=val_loader)
    except Exception as e:
        print("Trial %d failed with error: %s" % (trial.number, e))
        return float("inf")

    # --- 6. Return the Metric to Minimize ---
    val_loss = trainer.callback_metrics.get("val_loss_epoch",
                                            float("inf"))
    return val_loss


# ======== MAIN EXECUTION ========
if __name__ == "__main__":
    print("Starting Optuna study. Database at: %s" % OPTUNA_DB_PATH)
    study = optuna.create_study(
        direction="minimize",
        storage=OPTUNA_DB_PATH,
        study_name="stage2_autoencoder_tuning_v2",
        load_if_exists=True,
        pruner=optuna.pruners.MedianPruner(),
    )
    study.optimize(objective, n_trials=N_TRIALS)

    print("\n\n--- TUNING STUDY COMPLETE ---")
    print("Number of finished trials: %d" % len(study.trials))
    print("Best trial value: %.6f" % study.best_trial.value)
    print("Best Hyperparameters:")
    for key, value in study.best_trial.params.items():
        print("    %s: %s" % (key, value))


# Training

In [ ]:
import os
from pathlib import Path
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from torch.utils.data import DataLoader
import pytorch_lightning as pl

from config import project_paths, simd_r_drive_server_config
from models.pytorch.narrative_stack.stage2 import Stage2Autoencoder
from models.pytorch.narrative_stack.stage2.dataset import (
    Stage2StackDataset,
    stage2_collate_stacks,
)

# === CONFIG ===
OUTPUT_PATH = Path(project_paths.python_data / "stage2_training")
os.makedirs(OUTPUT_PATH, exist_ok=True)

EPOCHS = 15
PATIENCE = 5
BATCH_SIZE = 4
NUM_WORKERS = 2
LOOKUP_BATCH_SIZE = 32
GRAD_CLIP = 1.0

# === MODEL ===
model = Stage2Autoencoder()

# === DATALOADERS ===
train_loader = DataLoader(
    Stage2StackDataset(
        simd_r_drive_server_config=simd_r_drive_server_config,
        shuffle=True,
        lookup_batch_size=LOOKUP_BATCH_SIZE,
    ),
    batch_size=BATCH_SIZE,
    collate_fn=stage2_collate_stacks,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    num_workers=NUM_WORKERS,
)

val_loader = DataLoader(
    Stage2StackDataset(
        simd_r_drive_server_config=simd_r_drive_server_config,
        shuffle=False,
        lookup_batch_size=LOOKUP_BATCH_SIZE,
    ),
    batch_size=BATCH_SIZE,
    collate_fn=stage2_collate_stacks,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    num_workers=NUM_WORKERS,
)

# === CALLBACKS ===
early_stop_callback = EarlyStopping(
    monitor="val_loss_epoch",
    patience=PATIENCE,
    verbose=True,
    mode="min",
)

model_checkpoint = ModelCheckpoint(
    dirpath=OUTPUT_PATH,
    filename="stage2_checkpoint-epoch{epoch:02d}-val{val_loss_epoch:.4f}",
    monitor="val_loss_epoch",
    mode="min",
    save_top_k=3,  # Keep best 3
    every_n_epochs=1,
    verbose=True,
)

# === TRAINER ===
trainer = pl.Trainer(
    max_epochs=EPOCHS,
    logger=TensorBoardLogger(OUTPUT_PATH, name="stage2_autoencoder"),
    callbacks=[early_stop_callback, model_checkpoint],
    # accelerator="auto",
    accelerator="cpu",
    devices=1,
    gradient_clip_val=GRAD_CLIP,
)

# === TRAIN ===
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)


# Debug

## Hyperparameter Importance

In [ ]:
# file: analyze_study.py

import optuna
import os
from config import project_paths

# --- Configuration ---
OUTPUT_PATH = project_paths.python_data / "stage2_tuning"
OPTUNA_DB_PATH = f"sqlite:///{OUTPUT_PATH / 'stage2_study.db'}"
STUDY_NAME = "stage2_autoencoder_tuning_v2"

# --- Main Analysis ---
if __name__ == "__main__":
    print(f"Loading study '{STUDY_NAME}' from database at: {OPTUNA_DB_PATH}")
    study = optuna.load_study(study_name=STUDY_NAME, storage=OPTUNA_DB_PATH)

    print(f"Study loaded successfully with {len(study.trials)} trials.")
    print(f"Best trial value: {study.best_trial.value}")
    print("Best hyperparameters found:", study.best_trial.params)

    try:
        import optuna.visualization as vis
        fig = vis.plot_param_importances(study)
        print("\nGenerating plot... Close the plot window to exit the script.")
        fig.show()
    except Exception as e:
        print(f"Could not generate plot. Error: {e}")
        print("This can happen if the study has too few completed trials "
              "or if there's an issue with Plotly.")


## Tuning Latent Space Analysis

In [ ]:
import torch
import optuna
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA

from config import project_paths, simd_r_drive_server_config
from models.pytorch.narrative_stack.stage2.dataset import (
    Stage2StackDataset,
    stage2_collate_stacks,
)
from models.pytorch.narrative_stack.stage2 import Stage2Autoencoder


def load_trial_config_and_checkpoint(trial_number: int):
    db_path = project_paths.python_data / "stage2_tuning" / "stage2_study.db"
    study = optuna.load_study(
        study_name="stage2_autoencoder_tuning_v2",
        storage=f"sqlite:///{db_path}",
    )
    trial = study.trials[trial_number]

    cfg = {
        "loss_alpha": float(trial.params["loss_alpha"]),
        "dropout_rate": float(trial.params["dropout_rate"]),
        "weight_decay": float(trial.params["weight_decay"]),
        "warmup_steps": int(trial.params["warmup_steps"]),
        "encoder_latent_dim": int(trial.params["encoder_latent_dim"]),
        "num_latents": int(trial.params["num_latents"]),
        "encoder_depth": int(trial.params["encoder_depth"]),
        "shared_latent_dim": int(trial.params["shared_latent_dim"]),
        "query_dim": int(trial.params["query_dim"]),
        "decoder_depth": int(trial.params["decoder_depth"]),
        "batch_size": int(trial.params["batch_size"]),
        "lr": float(trial.params["lr"]),
    }

    ckpt_path = (
        project_paths.python_data
        / "stage2_tuning"
        / f"tuning_logs/trial_{trial_number}/checkpoints"
    )

    ckpt_file = next(ckpt_path.glob("*.ckpt"), None)
    if not ckpt_file or not ckpt_file.exists():
        raise FileNotFoundError(f"Checkpoint not found for trial {trial_number}")

    return cfg, ckpt_file


def plot_latents_from_trial(trial_number: int, num_samples: int = 512):
    cfg, ckpt_path = load_trial_config_and_checkpoint(trial_number)

    model = Stage2Autoencoder.load_from_checkpoint(str(ckpt_path), **cfg)
    model.eval()
    model.to("cuda" if torch.cuda.is_available() else "cpu")

    dataset = Stage2StackDataset(
        simd_r_drive_server_config=simd_r_drive_server_config,
        shuffle=True,
        lookup_batch_size=64,
    )
    dataloader = DataLoader(
        dataset,
        batch_size=cfg["batch_size"],
        collate_fn=stage2_collate_stacks,
        num_workers=2,
        persistent_workers=False,
    )

    latents = []
    with torch.no_grad():
        for batch in dataloader:
            stacks, masks, balances, periods = batch
            stacks = [s.to(model.device) for s in stacks]
            masks = [m.to(model.device) for m in masks]
            balances = [b.to(model.device) for b in balances]
            periods = [p.to(model.device) for p in periods]

            _, shared_latent = model(stacks, masks, balances, periods)
            latents.append(shared_latent.cpu())
            if sum(t.shape[0] for t in latents) >= num_samples:
                break

    latents = torch.cat(latents, dim=0)[:num_samples]

    # latents = latents[~torch.isnan(latents).any(dim=1)]  # Remove NaNs

    # print(latents)

    pca = PCA(n_components=3)
    latent_3d = pca.fit_transform(latents.numpy())

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(latent_3d[:, 0], latent_3d[:, 1], latent_3d[:, 2], s=10, alpha=0.7)
    ax.set_title(f"Shared Latent Space (PCA 3D) — Trial {trial_number}")
    plt.tight_layout()
    plt.show()

# Example usage
plot_latents_from_trial(14, num_samples=1024)
